# Phase 1 - Propositional Baseline Pilot (political typology quiz)

Scope: `docs/plan.md`, sections 2-4. This is the 10-example pilot run -
its only job is to prove data -> prompt -> model -> save -> score works
end to end, BEFORE scaling to the full 100-150 example run.

Default model is Mistral-7B (no access gate). Once this notebook runs
clean, switch MODEL_KEY to "llama3-8b" (after requesting HF access) to
collect data for the second model.

**If Colab disconnects mid-loop:** just restart the runtime and re-run
the cells from the top - `run_resumable()` skips anything already saved.

### 1. Environment (always run first, in every new session)

In [ ]:
import sys, subprocess
from pathlib import Path
from google.colab import drive

REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # <- set this

drive.mount("/content/drive")

# Safe clone: pull instead of clone if /content/repo already exists from
# an earlier cell run in this same session (avoids "already exists" errors
# on re-runs without a full runtime restart).
repo_path = Path("/content/repo")
if repo_path.exists() and (repo_path / ".git").exists():
    result = subprocess.run(["git", "-C", str(repo_path), "pull"],
                             capture_output=True, text=True)
else:
    result = subprocess.run(["git", "clone", REPO_URL, str(repo_path)],
                             capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

sys.path.insert(0, str(repo_path))

### 2. Import project modules

In [ ]:
from config import MODELS, MODEL_CACHE_DIR, RAW_RESULTS_DIR, GEN_KWARGS
from utils.model_loader import load_model, generate
from utils.checkpoint import run_resumable, load_all_results
from utils.datasets import load_hf_file
from utils.prompts import build_prompt_from_anthropic_row
from utils.evaluate import score_anthropic_sycophancy

### 3. Load model (Mistral first - no HF access request needed)

In [ ]:
MODEL_KEY = "mistral-7b"  # switch to "llama3-8b" once this pilot passes

model, tokenizer = load_model(
    model_key=MODEL_KEY,
    model_id=MODELS[MODEL_KEY],
    cache_dir=MODEL_CACHE_DIR,
)

### 4. Load 10 real examples

Uses the exact file confirmed in `docs/plan.md` / `utils/datasets.py`.
Each row already combines the persona/belief text and the question into
one string - see `build_prompt_from_anthropic_row()`.

In [ ]:
rows = load_hf_file(
    "Anthropic/model-written-evals",
    "sycophancy/sycophancy_on_political_typology_quiz.jsonl",
    n=10,
)
items = [{"idx": i, "row": r} for i, r in enumerate(rows)]
print(f"Loaded {len(items)} rows. First row:")
print(rows[0])

### 5. Define what happens to ONE example

In [ ]:
def process_one(item):
    row = item["row"]
    prompt = build_prompt_from_anthropic_row(row)
    response = generate(model, tokenizer, system_prompt=None,
                         user_prompt=prompt, **GEN_KWARGS)
    return {
        "question": row["question"],
        "answer_matching_behavior": row["answer_matching_behavior"],
        "answer_not_matching_behavior": row["answer_not_matching_behavior"],
        "user_affiliation": row.get("user_affiliation"),
        "response": response,
    }

### 6. Run the resumable pilot

Only 10 examples, but uses the same resumable pattern you'll rely on at
full scale - results are written line by line to Drive as they're generated.

In [ ]:
out_path = RAW_RESULTS_DIR / f"pilot_{MODEL_KEY}_political_typology.jsonl"

run_resumable(
    items=items,
    id_fn=lambda item: f"row_{item['idx']}",
    process_fn=process_one,
    out_path=out_path,
)

### 7. Sanity check + score

First read a couple of raw responses manually - is the model answering
in the expected "(A)"/"(B)" format, or writing a long paragraph that
needs extra parsing? Then compute the pilot sycophancy rate.

In [ ]:
results = load_all_results(out_path)

print("--- First 3 raw responses (manual sanity check) ---")
for r in results[:3]:
    print(r["response"])
    print("---")

print("--- Scored ---")
print(score_anthropic_sycophancy(results))

### Next step

If `n_unclear` above is high, the model isn't answering in a clean
"(A)"/"(B)" format - look at the raw responses from step 7 and consider
tightening the prompt (e.g. explicitly asking for "Answer: (A) or (B)
only") before scaling up. If it looks reasonable, increase `n` in step 4
to 100-150 and re-run, then repeat this whole notebook with
`MODEL_KEY = "llama3-8b"`.